# Operating patterns and leak-related sensor changes

Run the pipeline and dashboard export first. See [methodology](../docs/methodology.md) for protocol v2.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'plotly_mimetype'
ROOT = Path.cwd() if (Path.cwd()/'results').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT/'src'))
experiment = json.loads((ROOT/'results/experiment.json').read_text())
assert experiment['protocol_version'] == 2


## What daily operating pattern must a detector distinguish from leaks?

In [2]:
profiles = pd.read_csv(ROOT/'results/hourly_profiles.csv')
px.line(profiles, x='hour', y=['flow:p227','flow:p235','flow:PUMP_1'], labels={'value':'Flow (m³/h)'})

## Does the weekday pattern differ, and do weekly averages drift?

In [3]:
weekday = pd.read_csv(ROOT/'results/weekday_profiles.csv')
display(weekday[['weekday','flow:p227','flow:p235']])
weekly = pd.read_csv(ROOT/'results/weekly_means.csv')
px.line(weekly, x='timestamp', y=['flow:p227','flow:p235'], labels={'value':'Weekly mean flow (m³/h)'})

,weekday,flow:p227,flow:p235
0,0,113.300832,114.339552
1,1,114.111177,115.133172
2,2,114.151348,115.683656
3,3,114.460457,116.101126
4,4,112.846702,114.484396
5,5,108.757309,110.236690
6,6,109.432739,110.744254


## Which pressure and flow sensors move together?

In [4]:
correlation = pd.read_csv(ROOT/'results/correlations.csv', index_col=0)
px.imshow(correlation, zmin=-1, zmax=1, color_continuous_scale='BrBG')

## Inspect a focused distribution and pressure-flow relationship

In [5]:
statistics = pd.read_csv(ROOT/'results/sensor_statistics.csv')
display(statistics[statistics.sensor.isin(['pressure:n1','flow:p227','flow:p235'])])
x = pd.read_parquet(ROOT/'data/processed/telemetry.parquet', columns=['pressure:n1','flow:p227'])
px.scatter(x.loc['2019'].iloc[::48], x='flow:p227', y='pressure:n1', opacity=.4, labels={'flow:p227':'Flow (m³/h)','pressure:n1':'Pressure (m)'})

,sensor,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max,missing_pct,max_abs_change_5min
0,pressure:n1,210240.0,28.102923,0.547333,25.67,26.92,27.25,27.69,28.070,28.53,29.02,29.1700,29.36,0.0,0.51
33,flow:p227,210240.0,112.440700,35.850462,15.93,29.43,48.43,89.86,114.285,136.13,170.94,183.2261,205.44,0.0,44.85
34,flow:p235,210240.0,113.820067,33.838176,17.17,31.36,52.28,91.73,117.540,139.09,163.26,173.8400,195.28,0.0,50.16


## How do local before/onset/recovery summaries change around p523?

In [6]:
changes = pd.read_csv(ROOT/'results/event_sensor_changes.csv')
local = changes[changes['pipe'].eq('p523')].copy()
local['magnitude'] = local.change_in_pre_std.abs()
local.sort_values('magnitude', ascending=False).head(10).drop(columns='magnitude')

,pipe,sensor,type,pre_48h_median,onset_72h_median,change,change_in_pre_std,recovery_48h_median,recovery_available
538,p523,pressure:n506,abrupt,53.370,52.88,-0.490,-1.850917,53.300,True
522,p523,pressure:n105,abrupt,50.360,50.03,-0.330,-1.372169,50.305,True
523,p523,pressure:n114,abrupt,53.800,53.42,-0.380,-1.360504,53.735,True
536,p523,pressure:n469,abrupt,47.370,47.04,-0.330,-1.350711,47.320,True
539,p523,pressure:n516,abrupt,54.660,54.28,-0.380,-1.208271,54.580,True
537,p523,pressure:n495,abrupt,51.590,51.38,-0.210,-1.181466,51.540,True
530,p523,pressure:n332,abrupt,56.260,55.91,-0.350,-1.175254,56.165,True
541,p523,pressure:n549,abrupt,54.600,54.24,-0.360,-1.066328,54.500,True
552,p523,flow:p235,abrupt,97.655,121.74,24.085,0.942962,103.315,True
540,p523,pressure:n519,abrupt,47.400,47.11,-0.290,-0.709530,47.225,True


## Interpretation and limits

Daily demand is a major confounder. Correlation does not imply hydraulic adjacency or causation. The local comparison is descriptive: concurrent leaks, seasonality and unequal window coverage can change the summaries. A record-end event has no observed recovery.